# Qwen3-14B 회의록 요약 LoRA 학습 (Unsloth)
ModuStudy/Squiz - SSAFY L40S 46GB GPU

**D106팀 - GPU Device 3**

Unsloth로 2~5x 빠른 학습

## 1. GPU 설정 및 환경 확인

In [1]:
# SSAFY GPU 서버 설정 (D106팀 - Device 3)
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

# GPU 메모리 정리
import gc
import torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# GPU 확인
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
print(f"현재 GPU: {torch.cuda.get_device_name(0)}")

total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
used_mem = torch.cuda.memory_allocated(0) / 1024**3
free_mem = total_mem - used_mem

print(f"GPU 총 메모리: {total_mem:.1f} GB")
print(f"GPU 사용 중: {used_mem:.1f} GB")
print(f"GPU 여유: {free_mem:.1f} GB")

if free_mem < 30:
    print("\n⚠️ GPU 메모리 부족! 커널을 재시작하세요.")

CUDA 사용 가능: True
현재 GPU: NVIDIA L40S
GPU 총 메모리: 44.5 GB
GPU 사용 중: 0.0 GB
GPU 여유: 44.5 GB


## 2. Unsloth 설치 (최초 1회)

In [2]:
# ===== 설정 =====
MODEL_NAME = "unsloth/Qwen3-14B"  # Unsloth 최적화 버전
MAX_SEQ_LENGTH = 1024  # 데이터 최대 952 토큰이므로 충분

# LoRA 설정 (메모리 절약)
LORA_R = 32        # 64 → 32
LORA_ALPHA = 64    # 128 → 64
LORA_DROPOUT = 0.05

# 학습 하이퍼파라미터
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8  # effective batch size = 8
LEARNING_RATE = 1e-4
NUM_EPOCHS = 4
WARMUP_RATIO = 0.1

# 데이터 경로
DATA_PATH = "training_data_train.json"
VAL_PATH = "training_data_val.json"
OUTPUT_DIR = "./outputs/qwen3-14b-summarizer"

# 4-bit 양자화 사용
USE_4BIT = True

# 테스트용 입력 (Before/After 비교용)
TEST_INPUT = """다음 IT 스터디 회의 내용을 요약해주세요.

회의 내용:
김민수: 어 네 그럼 오늘 스터디 시작할게요. 오늘은 BFS DFS 문제 풀이 리뷰하기로 했죠?
이지은: 네네 맞아요. 저 백준 1260번 풀어왔는데요 음 DFS랑 BFS 둘 다 구현했거든요.
박준혁: 아 저도요. 근데 그 BFS에서 visited 체크 위치가 좀 헷갈리더라고요.
김민수: 아아 그거 중요한 포인트예요. 큐에 넣을 때 체크해야 중복 방문 안 해요.
이지은: 아 그렇구나. 저는 꺼낼 때 체크했거든요. 그래서 메모리 초과 났었어요.
박준혁: 오 그래서 그랬구나. 저도 같은 실수 했었는데.
김민수: 네 다음 주는 DP 문제 풀어오기로 해요. 백준 1149번 RGB거리요.
"""

print("설정 완료!")
print(f"모델: {MODEL_NAME}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")
print(f"LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")
print(f"Batch size: {BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} = {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Epochs: {NUM_EPOCHS}")

설정 완료!
모델: unsloth/Qwen3-14B
MAX_SEQ_LENGTH: 1024
LoRA rank: 32, alpha: 64
Batch size: 1 x 8 = 8
Epochs: 4


## 3. 설정값 정의

In [3]:
# ===== 설정 =====
MODEL_NAME = "unsloth/Qwen3-14B"  # Unsloth 최적화 버전
MAX_SEQ_LENGTH = 4096

# LoRA 설정
LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05

# 학습 하이퍼파라미터 (14B는 배치 줄임)
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4  # effective batch size = 8
LEARNING_RATE = 1e-4  # 14B는 약간 낮게
NUM_EPOCHS = 4
WARMUP_RATIO = 0.1

# 데이터 경로
DATA_PATH = "training_data_train.json"
VAL_PATH = "training_data_val.json"
OUTPUT_DIR = "./outputs/qwen3-14b-summarizer"

# 4-bit 양자화 사용
USE_4BIT = True

# 테스트용 입력 (Before/After 비교용)
TEST_INPUT = """다음 IT 스터디 회의 내용을 요약해주세요.

회의 내용:
김민수: 어 네 그럼 오늘 스터디 시작할게요. 오늘은 BFS DFS 문제 풀이 리뷰하기로 했죠?
이지은: 네네 맞아요. 저 백준 1260번 풀어왔는데요 음 DFS랑 BFS 둘 다 구현했거든요.
박준혁: 아 저도요. 근데 그 BFS에서 visited 체크 위치가 좀 헷갈리더라고요.
김민수: 아아 그거 중요한 포인트예요. 큐에 넣을 때 체크해야 중복 방문 안 해요.
이지은: 아 그렇구나. 저는 꺼낼 때 체크했거든요. 그래서 메모리 초과 났었어요.
박준혁: 오 그래서 그랬구나. 저도 같은 실수 했었는데.
김민수: 네 다음 주는 DP 문제 풀어오기로 해요. 백준 1149번 RGB거리요.
"""

print("설정 완료!")
print(f"모델: {MODEL_NAME}")
print(f"LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")
print(f"Batch size: {BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} = {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Epochs: {NUM_EPOCHS}")

설정 완료!
모델: unsloth/Qwen3-14B
LoRA rank: 64, alpha: 128
Batch size: 2 x 4 = 8
Epochs: 4


## 4. 데이터 로드

In [4]:
import json
from collections import Counter

# 학습 데이터 로드
with open(DATA_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(VAL_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)

print(f"학습 데이터: {len(train_data)}개")
print(f"검증 데이터: {len(val_data)}개")

# 타입별 분포 확인
type_counts = Counter([d['type'] for d in train_data])
print(f"\n타입별 분포:")
for t, c in type_counts.items():
    print(f"  - {t}: {c}개")

학습 데이터: 630개
검증 데이터: 70개

타입별 분포:
  - regular_summary: 451개
  - part_summary: 138개
  - integrated_summary: 41개


## 5. Unsloth 모델 로드

In [5]:
from unsloth import FastLanguageModel

# GPU 메모리 정리
gc.collect()
torch.cuda.empty_cache()

print(f"[MODEL] {MODEL_NAME} 로딩 중... (몇 분 소요)")

# Unsloth로 모델 로드 (4-bit 양자화 자동 적용)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # 자동 감지
    load_in_4bit=USE_4BIT,
)

print("모델 로드 완료!")
print(f"GPU 사용량: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[MODEL] unsloth/Qwen3-14B 로딩 중... (몇 분 소요)
==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA L40S. Num GPUs = 1. Max memory: 44.521 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

모델 로드 완료!
GPU 사용량: 10.37 GB


## 6. LoRA 설정 (Unsloth)

In [6]:
# Unsloth LoRA 적용
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth 최적화
    random_state=42,
)

# 학습 가능한 파라미터 확인
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"학습 가능 파라미터: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"전체 파라미터: {total_params:,}")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.1.4 patched 40 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


학습 가능 파라미터: 256,901,120 (2.91%)
전체 파라미터: 8,820,259,840


## 7. ⭐ Before 테스트 (파인튜닝 전)

In [7]:
def run_inference(model, tokenizer, test_input):
    """추론 실행 함수"""
    # 추론 모드로 전환
    FastLanguageModel.for_inference(model)
    
    prompt = f"""<|im_start|>system
당신은 IT 스터디 회의록을 요약하는 전문가입니다. 핵심 내용을 정확하게 정리합니다.<|im_end|>
<|im_start|>user
{test_input}<|im_end|>
<|im_start|>assistant
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    assistant_start = response.find("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
    assistant_end = response.find("<|im_end|>", assistant_start)
    summary = response[assistant_start:assistant_end] if assistant_end != -1 else response[assistant_start:]
    
    return summary.strip()


print("=" * 60)
print("🔵 BEFORE 테스트 (파인튜닝 전)")
print("=" * 60)
print("\n[입력]")
print(TEST_INPUT[:200] + "...")
print("\n[출력]")

before_result = run_inference(model, tokenizer, TEST_INPUT)
print(before_result)
print("\n" + "=" * 60)

🔵 BEFORE 테스트 (파인튜닝 전)

[입력]
다음 IT 스터디 회의 내용을 요약해주세요.

회의 내용:
김민수: 어 네 그럼 오늘 스터디 시작할게요. 오늘은 BFS DFS 문제 풀이 리뷰하기로 했죠?
이지은: 네네 맞아요. 저 백준 1260번 풀어왔는데요 음 DFS랑 BFS 둘 다 구현했거든요.
박준혁: 아 저도요. 근데 그 BFS에서 visited 체크 위치가 좀 헷갈리더라고요.
김민수: 아아 그거...

[출력]
<think>
Okay, I need to summarize this IT study meeting. Let me read through the conversation again to make sure I get all the key points.

First, the meeting starts with Kim Min-su confirming they're reviewing BFS and DFS problem solutions. Eui-jin mentions she solved problem 1260 on Baekjoon, implementing both DFS and BFS. Then Park Joon-hyuk says he also did it but had confusion about where to check visited in BFS. Kim explains that checking visited when adding to the queue is crucial to prevent revisiting, which Eui-jin realized she did at the wrong time (when popping from the queue), leading to a memory overflow. Park confirms he made the same mistake. Finally, Kim assigns next week's task as solving DP problem 1149 (RGB problem) on Baekjoon.

So 

## 8. 데이터셋 준비

In [8]:
from datasets import Dataset

def tokenize_function(examples):
    """토크나이징 + labels 마스킹 (assistant 응답만 학습)"""
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )

    # Labels 생성 (assistant 응답 부분만 학습)
    labels = result["input_ids"].copy()

    # assistant 마커 이전 부분 마스킹 (-100)
    text = examples["text"]
    assistant_marker = "<|im_start|>assistant\n"
    idx = text.find(assistant_marker)

    if idx != -1:
        prefix_text = text[:idx + len(assistant_marker)]
        prefix_tokens = tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
        mask_len = min(len(prefix_tokens), len(labels))
        for i in range(mask_len):
            labels[i] = -100

    result["labels"] = labels
    return result


# Dataset 생성
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Train dataset: {len(train_dataset)}개")
print(f"Val dataset: {len(val_dataset)}개")

# 토크나이징
print("\n토크나이징 중...")
train_dataset = train_dataset.map(tokenize_function, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(tokenize_function, remove_columns=val_dataset.column_names)

print("토크나이징 완료!")

Train dataset: 630개
Val dataset: 70개

토크나이징 중...


Map:   0%|          | 0/630 [00:00<?, ? examples/s]

Map:   0%|          | 0/70 [00:00<?, ? examples/s]

토크나이징 완료!


## 9. 학습 설정 (Unsloth Trainer)

In [9]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import UnslothTrainer

# 학습 모드로 전환
FastLanguageModel.for_training(model)

# 학습 설정
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    optim="adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
)

# Data Collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

# Unsloth Trainer
trainer = UnslothTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("Trainer 설정 완료!")
print(f"출력 디렉토리: {OUTPUT_DIR}")

Trainer 설정 완료!
출력 디렉토리: ./outputs/qwen3-14b-summarizer


## 10. 🚀 학습 시작!

In [10]:
print("=" * 60)
print("🚀 학습 시작! (Unsloth 최적화)")
print("=" * 60)
print(f"모델: {MODEL_NAME}")
print(f"데이터: {len(train_dataset)}개 (train) / {len(val_dataset)}개 (val)")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} = {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print("=" * 60)

# 학습!
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


🚀 학습 시작! (Unsloth 최적화)
모델: unsloth/Qwen3-14B
데이터: 630개 (train) / 70개 (val)
Epochs: 4
Batch size: 2 x 4 = 8


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 630 | Num Epochs = 4 | Total steps = 316
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 256,901,120 of 15,025,208,320 (1.71% trained)


OutOfMemoryError: CUDA out of memory. Tried to allocate 98.00 MiB. GPU 0 has a total capacity of 44.52 GiB of which 24.25 MiB is free. Including non-PyTorch memory, this process has 44.49 GiB memory in use. Of the allocated memory 33.63 GiB is allocated by PyTorch, and 10.33 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 11. 모델 저장

In [ ]:
# LoRA 어댑터 저장
lora_output_dir = f"{OUTPUT_DIR}/lora"
model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)

print(f"\nLoRA 어댑터 저장 완료: {lora_output_dir}")
print("\n학습 완료! 🎉")

## 12. ⭐ After 테스트 (파인튜닝 후)

In [ ]:
print("=" * 60)
print("🟢 AFTER 테스트 (파인튜닝 후)")
print("=" * 60)
print("\n[입력]")
print(TEST_INPUT[:200] + "...")
print("\n[출력]")

after_result = run_inference(model, tokenizer, TEST_INPUT)
print(after_result)
print("\n" + "=" * 60)

## 13. 📊 Before vs After 비교

In [ ]:
print("=" * 60)
print("📊 Before vs After 비교")
print("=" * 60)

print("\n🔵 [BEFORE - 파인튜닝 전]")
print("-" * 40)
print(before_result)

print("\n🟢 [AFTER - 파인튜닝 후]")
print("-" * 40)
print(after_result)

print("\n" + "=" * 60)
print("✅ 학습 및 테스트 완료!")
print("=" * 60)

## 14. LoRA 병합 및 GGUF 변환

In [ ]:
# LoRA 병합된 전체 모델 저장 (GGUF 변환용)
merged_output_dir = "./models/qwen3-14b-summarizer-merged"

print("LoRA 병합 중...")
model.save_pretrained_merged(
    merged_output_dir,
    tokenizer,
    save_method="merged_16bit",  # FP16으로 저장
)
print(f"병합된 모델 저장 완료: {merged_output_dir}")

In [ ]:
# GGUF 직접 변환 (Unsloth 기능)
gguf_output_dir = "./models"

print("GGUF Q4_K_M 변환 중...")
model.save_pretrained_gguf(
    gguf_output_dir,
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"GGUF 변환 완료: {gguf_output_dir}")
print("\n파일 확인:")
!ls -lh ./models/*.gguf